## Skip-gram を実装する
GPU の設定を推奨

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import collections
import random
import json

! pip install mecab-python3 unidic-lite
import MeCab

# 描画
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [ ]:
# load data 
def read_dataset(file_path):
    # 複数の行に分かれた日本語文を読み込み，単語ごとに分割してリストに格納し，generatorで返す
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            # MeCabを用いて単語に分割
            mecab = MeCab.Tagger("-Owakati")
            words = mecab.parse(line).strip().split()
            yield words

# データセットの読み込み
data = list(read_dataset("data.txt"))
print(data[:2])

In [ ]:
# 単語をIDに変換するための辞書を作成
word2id = collections.defaultdict(lambda: len(word2id))
SENT_BOUNDARY = word2id["<S>"] # 文の開始を表す特殊なトークン
UNK = word2id["<UNK>"] # 語彙に存在しない単語を表す特殊なトークン

for sentence in data:
    for word in sentence:
        word2id[word] # 単語をIDに変換してword2idに登録

word2id = collections.defaultdict(lambda: UNK, word2id) # 存在しない単語はUNKにマッピング

# 保存用に IDから単語へのマッピングも作成
id2word = {i: w for w, i in word2id.items()}

# 語彙のサイズを取得
vocab_size = len(word2id)

# 処理結果の表示
print(f"Vocabulary size: {vocab_size}")
print(f"word2id: {dict(list(word2id.items())[:10])}")
print(f"存在しない単語を入力するとUNKにマッピングされる例: {word2id['存在しない単語']}")

# 辞書の保存
with open("word2id.json", "w", encoding="utf-8") as f:
    json.dump(dict(word2id), f, ensure_ascii=False, indent=4)


In [ ]:
# ハイパーパラメータ
EMBEDDING_DIM = 10 # 単語の埋め込みベクトルの次元数
NEGATIVE_SAMPLES = 3 # ネガティブサンプリングの数
WINDOW_SIZE = 2 # 隣接 N 単語の数
EPOCHS = 40 # 学習のエポック数

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# モデルの定義
class Word2VecModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(Word2VecModel, self).__init__()
        # 中心単語用行列 v
        self.v_embeddings = nn.Embedding(vocab_size, embedding_dim)
        # 文脈用行列 u
        self.u_embeddings = nn.Embedding(vocab_size, embedding_dim)

    def forward_target(self, target_ids): # 中心単語のIDを入力として，対応する埋め込みベクトルを出力
        return self.v_embeddings(target_ids)

    def forward_context(self, context_ids): # 文脈単語のIDを入力として，対応する埋め込みベクトルを出力
        return self.u_embeddings(context_ids)

In [ ]:
# 目的関数
def negative_sampling_loss(v_embed, u_pos_embed, u_neg_embed):
    """
    v_embed: [batch, dim] 中心単語の埋め込みベクトル
    u_pos_embed: [batch, dim] 正解文脈単語の埋め込みベクトル
    u_neg_embed: [batch, neg, dim] 負例文脈単語の埋め込みベクトル
    """
    # 正解ペアのスコア: log(sigmoid(v · u_pos))
    pos_score = torch.sum(v_embed * u_pos_embed, dim=1)
    pos_loss = torch.log(torch.sigmoid(pos_score))

    # 負例ペアのスコア: Σ log(sigmoid(-v · u_neg))
    # [batch, 1, dim] * [batch, dim, neg] -> [batch, 1, neg]
    neg_score = torch.bmm(v_embed.unsqueeze(1), u_neg_embed.transpose(1, 2)).squeeze(1)
    neg_loss = torch.log(torch.sigmoid(-neg_score)).sum(dim=1)

    # 対数をとっているので，割り算ではなく足し算で損失を計算
    return -(pos_loss + neg_loss).mean()

In [ ]:
# 学習の準備
model = Word2VecModel(vocab_size, EMBEDDING_DIM).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.005) # 勾配法の設定

In [ ]:
# 学習データの作成
pairs = []
for sentence in data:
    for i, word in enumerate(sentence):
        target_id = word2id[word]
        for j in range(i - WINDOW_SIZE, i + WINDOW_SIZE + 1):
            # 中心単語と同じ位置の単語は文脈単語として考慮しない
            if i == j:
                continue
            # 文の境界を考慮して文脈単語のIDを取得
            context_id = word2id[sentence[j]] if 0 <= j < len(sentence) else SENT_BOUNDARY
            pairs.append((target_id, context_id))

print(f"学習データの例: {pairs[:10]}")

In [ ]:
# 学習ループ
for epoch in range(EPOCHS):
    total_loss = 0
    for target, pos_context in pairs:
        # データの準備
        t_tensor = torch.LongTensor([target]).to(device)
        p_tensor = torch.LongTensor([pos_context]).to(device)
        
        # ネガティブサンプリング
        neg_indices = []
        while len(neg_indices) < NEGATIVE_SAMPLES:
            idx = random.randint(0, vocab_size - 1)
            # 生成された負例のIDが正解文脈単語のIDと異なることを確認
            if idx != pos_context:
                neg_indices.append(idx)
        n_tensor = torch.LongTensor([neg_indices]).to(device)

        # 順伝播 (ベクトル抽出)
        v_e = model.forward_target(t_tensor)
        u_p_e = model.forward_context(p_tensor)
        u_n_e = model.forward_context(n_tensor)

        # 目的関数の計算
        loss = negative_sampling_loss(v_e, u_p_e, u_n_e)

        # 最適化
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d}, Loss: {total_loss/len(pairs):.4f}")

In [ ]:
# モデルの保存
torch.save({
    "model_state_dict": model.state_dict(),
    "word2id": dict(word2id),
    "id2word": id2word
}, "skip_gram_model.pth")

In [ ]:
# 推論テスト：語彙にある単語を，動的に描画してみる．

# 日本語文字に対応させるために，フォントを設定
plt.rcParams["font.family"] = "IPAexGothic"

# 埋め込みベクトルの抽出
embeddings = model.v_embeddings.weight.data.cpu().numpy()
# PCAで次元削減
pca = PCA(n_components=2)
reduced_embeddings = pca.fit_transform(embeddings)
plt.figure(figsize=(8, 8))
for i in range(vocab_size):
    plt.scatter(reduced_embeddings[i, 0], reduced_embeddings[i, 1])
    plt.text(reduced_embeddings[i, 0]+0.01, reduced_embeddings[i, 1]+0.01, id2word[i], fontsize=9)
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.grid()
plt.show()